# Clean and prepare mine locations

Loads the raw mine database, assigns sequential IDs, and exports:
- `data/output/cmr-mine-locations/all_mines_with_id.csv` — used by the path-generator
- `data/output/cmr-mine-locations/all_mines_with_id.gpkg` — GeoPackage for QGIS inspection

Run this notebook once before running `model/02_path-generator-run.py`.

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

REPO_ROOT   = Path("..").resolve()
INPUT_FP    = REPO_ROOT / "data/output/cmr-mine-locations/all_mines_df.csv"
OUTPUT_CSV  = REPO_ROOT / "data/output/cmr-mine-locations/all_mines_with_id.csv"
OUTPUT_GPKG = REPO_ROOT / "data/output/cmr-mine-locations/all_mines_with_id.gpkg"

In [ ]:
all_mines_df = pd.read_csv(INPUT_FP)
print(f"Loaded {len(all_mines_df)} mines")
all_mines_df.head()

In [ ]:
print(f"Number of mines: {len(all_mines_df)}")
print(f"\nNumber of mines at each stage:")
print(all_mines_df["DEV_STAGE_AGGREGATED_SNL"].value_counts())

In [ ]:
# Sort by development stage (custom order), add sequential ID as first column, and save
stage_order = ["Early-stage", "Late-stage", "Mine-stage"]

all_mines_df = all_mines_df.copy()
all_mines_df["DEV_STAGE_AGGREGATED_SNL"] = pd.Categorical(
    all_mines_df["DEV_STAGE_AGGREGATED_SNL"],
    categories=stage_order,
    ordered=True,
)

all_mines_df = all_mines_df.sort_values(
    by=["DEV_STAGE_AGGREGATED_SNL", "PROP_NAME"],
    na_position="last",
).reset_index(drop=True)

all_mines_df.insert(0, "ID", [f"cmr_{i:03d}" for i in range(1, len(all_mines_df) + 1)])

all_mines_df["DEV_STAGE_AGGREGATED_SNL"] = all_mines_df["DEV_STAGE_AGGREGATED_SNL"].cat.rename_categories(
    {"Mine-stage": "Built"}
)

all_mines_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV}")
all_mines_df.head()

In [ ]:
# Convert mines CSV to GeoPackage with geometry
mines_gdf = gpd.GeoDataFrame(
    all_mines_df,
    geometry=gpd.points_from_xy(all_mines_df["LONGITUDE"], all_mines_df["LATITUDE"]),
    crs="EPSG:4326",
)

mines_gdf.to_file(OUTPUT_GPKG, driver="GPKG")
print(f"Saved: {OUTPUT_GPKG}")
mines_gdf.head()